# Test VITS2 6K Checkpoint

Generate the Arjun & Priya podcast from your saved 6K step checkpoint.

In [ ]:
# Step 1: Setup
import os
os.chdir('/content')

# Clone repo
if not os.path.exists('/content/indian_tts'):
    !git clone https://github.com/seetha0712/text2speech_1.git /content/indian_tts

os.chdir('/content/indian_tts')
!git checkout claude/custom-indian-tts-model-TUAjJ
!git pull origin claude/custom-indian-tts-model-TUAjJ
!pip install -q -r requirements.txt
!pip install -q -e .
!apt-get install -qq espeak-ng > /dev/null 2>&1
print("Setup complete!")

In [ ]:
# Step 2: Find your checkpoint
# Check both local and Google Drive
import glob

from google.colab import drive
drive.mount('/content/drive')

# Search for checkpoints
locations = [
    '/content/outputs/checkpoints/checkpoint_*.pt',
    '/content/drive/MyDrive/indian_tts_checkpoints/checkpoint_*.pt',
]

found = []
for pattern in locations:
    ckpts = sorted(glob.glob(pattern))
    for c in ckpts:
        size_mb = os.path.getsize(c) / 1e6
        print(f"  Found: {c} ({size_mb:.0f} MB)")
        found.append(c)

if not found:
    print("No checkpoints found! Upload your checkpoint_step_6000.pt to Google Drive.")
else:
    checkpoint = found[-1]
    print(f"\nUsing: {checkpoint}")

In [ ]:
# Step 3: Generate test sentences
from indian_tts.inference import IndianTTS
import IPython.display as ipd
import numpy as np
import soundfile as sf

tts = IndianTTS(checkpoint)

test_texts = [
    "Hello, welcome to our Indian text to speech system.",
    "The weather in Bangalore is very pleasant today.",
    "India has over three hundred AI startups and growing.",
]

for text in test_texts:
    print(f"\n--- {text} ---")
    for voice in ['male', 'female']:
        audio = tts.synthesize(text, voice=voice)
        print(f"[{voice.upper()}]")
        ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))

In [ ]:
# Step 4: Generate full podcast
os.chdir('/content/indian_tts')
!python -m indian_tts.podcast_demo \
    --checkpoint {checkpoint} \
    --output /content/outputs/vits2_6k_podcast

In [ ]:
# Step 5: Listen
podcast_path = '/content/outputs/vits2_6k_podcast/podcast_full.wav'
if os.path.exists(podcast_path):
    print("VITS2 @ 6K steps — Full Podcast:")
    ipd.display(ipd.Audio(podcast_path))
    
    # Also play first few lines
    for f in sorted(glob.glob('/content/outputs/vits2_6k_podcast/line_*.wav'))[:4]:
        print(f"\n{os.path.basename(f)}")
        ipd.display(ipd.Audio(f))
else:
    print("Podcast not generated. Check errors above.")

# Save to Drive
import shutil
drive_dir = '/content/drive/MyDrive/indian_tts_checkpoints'
os.makedirs(drive_dir, exist_ok=True)
if os.path.exists(podcast_path):
    shutil.copy2(podcast_path, os.path.join(drive_dir, 'vits2_6k_podcast.wav'))
    print(f"\nSaved to Drive: {drive_dir}/vits2_6k_podcast.wav")